# CP6 - LangGraph investigation agent (Mode 2, D4) - Colab run in four parts

This notebook runs the CP6 agent with **free, open language models served locally by Ollama** on a free Colab GPU. The full
run is four agent runs (2 arms x 2 models), too long for one Colab session, so it is split into **four parts, one per session**:

| PART | Arm | Model | Why this order |
|---|---|---|---|
| 1 | A2 (agent, no graph tools) | `llama3.1-8b` | the main comparison; llama has a documented training cutoff |
| 2 | A2 | `qwen2.5-coder-7b` | the main comparison, second model |
| 3 | A3 (agent + callers/callees) | `llama3.1-8b` | the ablation |
| 4 | A3 | `qwen2.5-coder-7b` | the ablation, second model |

**How to use it:** before the first session, upload `cp6_graph_tables.zip` to Google Drive as
`MyDrive/vulntrace_cp6/cp6_graph_tables.zip`. Then, in each session: set `PART` in the first code cell, choose
`Runtime -> Change runtime type -> T4 GPU`, and `Runtime -> Run all`. If a session is cut off before its part finishes, start a
new session and run **the same PART again**: finished cases come back from the traces and the response cache on Drive, so it
continues where it stopped. Go to the next PART only when the last cell says this part is complete.

**What happens in a session:** clone the repository, install Go and the project. The **first** session rebuilds the D4 evidence on
the local Colab disk (about 25 minutes), stops if it does not match the committed case list, builds the commit diffs, unpacks the
neighbour tables, and saves all of it to Drive as one archive (`d4_evidence.tar.gz`). Later sessions restore that archive in a minute
and pass the same gate. Then: embed the D4 chunks (cached on Drive), install Ollama, pull **this part's** model, check that it
really generates text, run **this part's** agent arm, and zip all results so far.

**Zero cost:** only open models run, on the Colab machine, through `http://localhost:11434/v1`. The project's endpoint rule refuses
any non-local host and every known paid API host. No paid API is called.
**What the results zip contains:** result JSON (answers, stop reasons, counts, tokens, times - no prompts, no code, no raw model
replies) and the gzipped agent traces (actions, raw model replies, IDs shown, token counts - no prompts and no tool output text). The
traces are for the failure analysis in the next Claude Code session; they are not committed to the repository.
**Evidence on Drive:** `d4_evidence.tar.gz` holds the rebuilt evidence (code of public open-source repositories, read with git). It
stays private in your Drive and never goes into the results zip.

**Running time:** not measured yet. Rough estimate from the CP5 run (about 5 s per model call on a T4, 6-12 calls per case): about
1-2 hours per part, plus about 35 minutes of setup in the first session and about 10 minutes in later ones. The timings written by
the last cell are the real record; please keep them.

In [ ]:
# Settings. PART chooses this session's run (see the table above): 1 = A2 llama, 2 = A2 qwen, 3 = A3 llama, 4 = A3 qwen.
# Re-run the same PART after a disconnect; go on only when the last cell says the part is complete.
# DRY_RUN=True swaps the language model and the embedder for deterministic fakes (a quick pipeline check, no Ollama).
PART = 1
BRANCH = "claude/gracious-carson-ch1i9r"
REPO_URL = "https://github.com/AMAN-PRIYANSH/REPO-MIND.git"
DRY_RUN = False
MODELS = ["llama3.1-8b", "qwen2.5-coder-7b"]      # the two CP5 models (aliases in the project's models_config)
ARMS = ["A2", "A3"]                               # A0 and A1 need no run: A0 is in every agent row, A1 comes from CP5's results
PARTS = {1: ("A2", "llama3.1-8b"), 2: ("A2", "qwen2.5-coder-7b"), 3: ("A3", "llama3.1-8b"), 4: ("A3", "qwen2.5-coder-7b")}
ARM, MODEL = PARTS[PART]
GO_VERSION = "1.24.7"          # rebuilds the D4 evidence exactly as CP4/CP5 did
GRAPH_GO_VERSION = "1.26.10"   # only for rebuilding neighbour tables here if the zip is missing (many go.mod files need Go >= 1.25)
CONTEXT_TOKENS = 32768         # Ollama context length; the agent's own guard stops before 24000 prompt tokens per call
LIMIT = None                   # None = all 120 D4 cases; a number runs only the first N (a trial)
REBUILD_EVIDENCE = False       # True ignores the evidence archive on Drive and rebuilds it (only if the gate keeps failing)
print(f"PART {PART}: arm {ARM}, model {MODEL}")

In [ ]:
# A tiny timer so the real running time of every step is recorded (kept on Drive across sessions, included in the results zip).
import json, os, shutil, subprocess, sys, time
class Timings:
    def __init__(self): self.rows, self.t0, self.label = [], None, None
    def start(self, label): self.label, self.t0 = label, time.time(); print(">>>", label)
    def stop(self):
        sec = round(time.time() - self.t0, 1); self.rows.append({"part": PART, "step": self.label, "seconds": sec}); print(f"<<< {self.label}: {sec} s")
T = Timings()

In [ ]:
# Check whether a GPU is attached. Ollama uses it automatically; without one the 7-8B models are far too slow for the run.
GPU = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU available:", GPU)
if GPU:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
DEVICE = "cuda" if GPU else "cpu"
if not GPU and not DRY_RUN:
    print("WARNING: no GPU. Switch the runtime to T4 GPU, or set DRY_RUN = True for a pipeline check only.")

In [ ]:
# Mount Google Drive for the files worth keeping across sessions: the evidence archive, embeddings, the LLM answer cache, agent
# traces, timings and results. Git clones stay on the LOCAL disk (git behaves badly on the Drive mount).
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/vulntrace_cp6"       # Drive: evidence archive, LLM cache, traces, results
os.makedirs(WORK, exist_ok=True)
CP5_EMB = "/content/drive/MyDrive/vulntrace_cp5/embeddings"   # the CP5 run's bge-small cache, reused if it is there
EMB = CP5_EMB if os.path.isdir(CP5_EMB) else f"{WORK}/embeddings"
CACHE = "/content/cache"                            # local disk: git clones + vulndb snapshot + Go module cache
DATA = "/content/data"                              # local disk: D4 evidence and the agent evidence
os.makedirs(CACHE, exist_ok=True)
os.environ["VULNTRACE_CACHE"] = CACHE
SUFFIX = "_dryrun" if DRY_RUN else ""
OUT = f"{WORK}/results/cp6{SUFFIX}"
LLM_CACHE = f"{WORK}/llm_cache{SUFFIX}"
TRACES = f"{WORK}/traces{SUFFIX}"
GRAPH_ZIP = f"{WORK}/cp6_graph_tables.zip"          # upload this before the first session (see the CP6 report)
EVIDENCE_TAR = f"{WORK}/d4_evidence.tar.gz"         # written by the first session, restored by the later ones
TIMINGS = f"{WORK}/timings{SUFFIX}.json"
DRY = "--dry-run" if DRY_RUN else ""
LIM = f"--limit {LIMIT}" if LIMIT else ""
os.makedirs(OUT, exist_ok=True)
print("Drive:", WORK, "| embeddings:", EMB, "| local evidence:", DATA, "| results:", OUT)
print("neighbour-table zip on Drive:", os.path.exists(GRAPH_ZIP), "| evidence archive on Drive:", os.path.exists(EVIDENCE_TAR))

In [ ]:
# Clone (or update) the repository at the CP6 branch. The code that does the work lives in the repository; this notebook only calls it.
REPO = "/content/REPO-MIND"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/REPO-MIND

In [ ]:
# Install Go (Colab does not ship it). The project's small Go helper parses Go files to find function boundaries while the evidence
# is rebuilt. Installed under /content only, nothing system-wide. GRAPH_GO_VERSION is used only by the fallback neighbour-table rebuild.
def install_go(version):
    if not os.path.exists(f"/content/go{version}/go/bin/go"):
        subprocess.run(["wget", "-q", f"https://go.dev/dl/go{version}.linux-amd64.tar.gz", "-O", "/content/go.tgz"], check=True)
        os.makedirs(f"/content/go{version}", exist_ok=True)
        subprocess.run(["tar", "-C", f"/content/go{version}", "-xzf", "/content/go.tgz"], check=True)
    return f"/content/go{version}/go/bin"
GO_BIN = install_go(GO_VERSION)
BASE_PATH = os.environ["PATH"]
os.environ["PATH"] = GO_BIN + ":" + BASE_PATH
os.environ["GOTOOLCHAIN"] = "local"
os.environ["CGO_ENABLED"] = "0"
print(subprocess.run(["go", "version"], capture_output=True, text=True).stdout)

In [ ]:
# Install this project with the [ml] extra (sentence-transformers for the bge-small embedder; Colab already has torch with CUDA)
# and the [agent] extra (LangGraph, pinned in pyproject.toml).
!pip install -q -e ".[ml,agent]"
# The notebook's Python session does not see the new install until it is restarted, so point it at the checkout directly.
import sys, importlib
sys.path.insert(0, "/content/REPO-MIND/src")
importlib.invalidate_caches()
!python -m vulntrace doctor

In [ ]:
# Fetch the pinned golang/vulndb snapshot (the advisory texts) and the Go helper's dependencies (both quick).
T.start("fetch vulndb + build helper")
!python -m vulntrace dataset fetch
!python -m vulntrace helper fetch
!python -m vulntrace helper build
T.stop()

In [ ]:
# D4 evidence: restore the archive a previous session saved on Drive (about a minute), or - in the first session - rebuild the
# release ranges and candidate code for the 120 committed case IDs on the local disk (network-heavy; about 25 minutes).
import tarfile
RESTORED = False
if os.path.exists(EVIDENCE_TAR) and not REBUILD_EVIDENCE:
    T.start("restore D4 evidence from Drive")
    with tarfile.open(EVIDENCE_TAR) as tf:
        tf.extractall(DATA, filter="data")
    RESTORED = True
    T.stop()
else:
    T.start("materialize D4")
    !python -u -m vulntrace dataset materialize --set d4 --data-dir "{DATA}" --summary-out "{WORK}/D4_materialize.json"
    T.stop()
print("evidence restored from Drive:", RESTORED)

In [ ]:
# GATE: stop here if the evidence (rebuilt or restored) does not match the committed case list (more than 3 cases with a different
# status, or no candidate functions at all). If this cell stops the notebook, do NOT continue: set REBUILD_EVIDENCE = True and run all
# again, or run  !python -m vulntrace dataset diagnose --case GO-2020-0038  and read the git errors.
from collections import Counter
from vulntrace.eval import evidence_gate
case_list = json.load(open("docs/results/datasets/D4_cases.json"))
statuses, n_candidates = evidence_gate.evidence_statuses(case_list, DATA, "release_range")
problems = evidence_gate.check_evidence(case_list, statuses, n_candidates)
print("D4 statuses:", dict(Counter(statuses.values())), "| expected:", case_list.get("status_counts"), "| candidates:", n_candidates)
if problems:
    raise SystemExit("STOP - the evidence is not usable: " + "; ".join(problems))
print("gate passed")

In [ ]:
# Agent evidence 1: the Go diff of every commit of every release range (for the show_commit tool), read from the local git cache with
# `git show` (read-only; nothing is executed). About 2 minutes. Skipped when the evidence came from the Drive archive (it holds them).
if not RESTORED:
    T.start("agent prepare --diffs")
    !python -u -m vulntrace agent prepare --set d4 --diffs --data-dir "{DATA}" --summary-out "{WORK}/agent_evidence_colab.json"
    T.stop()

In [ ]:
# Agent evidence 2: the call-graph neighbour tables for arm A3 (one-hop callers and callees of every candidate). Preferred: unzip the
# tables the build session made (GRAPH_ZIP on Drive). Otherwise rebuild them here with Go GRAPH_GO_VERSION and the Go module proxy -
# CPU-heavy, an hour or more on Colab's 2 CPUs. Either way they are compared with the committed summary
# docs/results/cp6/agent_evidence.json; a case without a table runs A3 without the graph tools and is counted.
import zipfile, glob
from vulntrace.agent import prepare as agent_prepare
if len(glob.glob(f"{DATA}/agent/graph/*.json.gz")) >= 118:
    print("neighbour tables already in place (from the evidence archive)")
elif os.path.exists(GRAPH_ZIP):
    with zipfile.ZipFile(GRAPH_ZIP) as z:
        for name in z.namelist():
            if not name.startswith("agent/graph/") or not name.endswith(".json.gz") or ".." in name:
                raise SystemExit(f"STOP - unexpected file in {GRAPH_ZIP}: {name}")
        z.extractall(DATA)
    print("neighbour tables unpacked from", GRAPH_ZIP)
else:
    print("WARNING: no", GRAPH_ZIP, "- rebuilding the neighbour tables here (slow). Upload the zip to skip this.")
    T.start("agent prepare --graph (rebuild in Colab)")
    GRAPH_GO_BIN = install_go(GRAPH_GO_VERSION)
    os.environ["PATH"] = GRAPH_GO_BIN + ":" + BASE_PATH
    !python -m vulntrace helper build
    !python -u -m vulntrace agent prepare --set d4 --graph --jobs 1 --data-dir "{DATA}" --gomodcache /content/cache/gomod --summary-out "{WORK}/agent_evidence_colab.json"
    os.environ["PATH"] = GO_BIN + ":" + BASE_PATH
    T.stop()
committed = json.load(open("docs/results/cp6/agent_evidence.json"))
cmp = agent_prepare.compare_with_summary([c["id"] for c in case_list["cases"]], DATA, committed)
for kind, res in cmp.items():
    print(kind, {k: len(v) for k, v in res.items()}, "| different:", res["different"][:10])

In [ ]:
# Save the evidence to Drive once (first session only), so later sessions restore it in a minute instead of rebuilding it.
# The archive holds release ranges, candidate code, commit diffs and neighbour tables; it stays on your Drive only.
if not RESTORED and not os.path.exists(EVIDENCE_TAR):
    T.start("save evidence archive to Drive")
    tmp = "/content/d4_evidence.tar.gz"
    with tarfile.open(tmp, "w:gz") as tf:
        for sub in ("release_range", "candidate_code", "agent/commit_diffs", "agent/graph"):
            if os.path.isdir(f"{DATA}/{sub}"):
                tf.add(f"{DATA}/{sub}", arcname=sub)
    shutil.copy(tmp, EVIDENCE_TAR)
    print(EVIDENCE_TAR, os.path.getsize(EVIDENCE_TAR) // (1024 * 1024), "MiB")
    T.stop()

In [ ]:
# Embed every D4 chunk (code+commits variant, the one the fixed retriever uses) with bge-small, caching vectors on Drive by the chunk
# text's hash (the CP5 cache is reused when present, so this is quick). Skipped in a dry run.
if not DRY_RUN:
    T.start("embed D4 chunks (bge-small)")
    !python -u -m vulntrace index build --set d4 --embedder bge-small --variant "code+commits" --data-dir "{DATA}" --embedding-cache "{EMB}" --device {DEVICE}
    T.stop()

In [ ]:
# Install and start Ollama on this machine (listening on 127.0.0.1 only) with a 32k-token context, then wait until it answers.
# Skipped in a dry run. zstd is needed by Ollama's installer. `which ollama` only proves the CLI exists (a partial install has no
# runner and fails every generate call with HTTP 500), so the pull cell below also makes one tiny generate call and, if that fails,
# calls reinstall_ollama() once before the run goes on.
import urllib.request
def ollama_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3).read(); return True
    except Exception:
        return False
def start_ollama():
    env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_CONTEXT_LENGTH=str(CONTEXT_TOKENS))
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_up(): break
        time.sleep(2)
def reinstall_ollama():
    subprocess.run("pkill -f 'ollama serve'", shell=True); time.sleep(2)
    subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    start_ollama()
if not DRY_RUN:
    T.start("install + start Ollama")
    if shutil.which("ollama") is None:
        !apt-get -qq install -y zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh
    if not ollama_up():
        start_ollama()
    print("ollama answering:", ollama_up())
    print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout)
    T.stop()

In [ ]:
# Pull THIS PART's model and print the exact tag and digest the server reports. A model that cannot be pulled stops the part - there is
# no fallback to any other service. Then the install check: one tiny generate call; on failure reinstall once; if it still fails the
# notebook stops here (OllamaSetupError) instead of failing call by call.
from vulntrace.llm import models_config, ollama_setup
MODEL_INFO = {}
if not DRY_RUN:
    tag = models_config.resolve(MODEL)["ollama_tag"]
    T.start(f"pull {tag}")
    r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)
    T.stop()
    if r.returncode != 0:
        json.dump({MODEL: (r.stderr or r.stdout)[-500:]}, open(f"{OUT}/model_problems_part{PART}.json", "w"), indent=1)
        raise SystemExit(f"STOP - could not pull {tag}: {(r.stderr or r.stdout)[-300:]}")
    ollama_setup.ensure_working(lambda: ollama_setup.probe_generate(tag), reinstall_ollama)
    tags = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]
    MODEL_INFO = {m["name"]: {"digest": m["digest"], "size": m["size"], "details": m.get("details")} for m in tags}
    print(json.dumps(MODEL_INFO, indent=1))
RUN_MODEL = "fake" if DRY_RUN else MODEL
print("this part runs: arm", ARM, "model", RUN_MODEL)

In [ ]:
# THIS PART's agent run: the agent investigates every D4 case with at most 8 steps and submits a ranked answer of 1-3 candidate IDs
# (or abstains). The result file OUT/agent__<arm>__<model>.json is written only when every case is done; until then each finished case
# is a gzipped trace in TRACES on Drive. If the session is cut off, run the same PART again: finished cases are not run again.
# Three cases in a row with a server error stop the run (re-run the notebook once the server works).
T.start(f"agent {ARM} {RUN_MODEL}")
!python -u -m vulntrace eval symbols --strategy agent --set d4 --arm {ARM} --model {RUN_MODEL} {DRY} {LIM} --data-dir "{DATA}" --embedding-cache "{EMB}" --device {DEVICE} --llm-cache "{LLM_CACHE}" --trace-dir "{TRACES}" --out "{OUT}"
T.stop()

In [ ]:
# Render the results document from all result files written so far (every finished part), to look at it here. The next Claude Code
# session renders the committed copy the same way from the zip (with the CP5 S1 files for arm A1, which are in the repository).
!python -m vulntrace results render --doc cp6 --input "{OUT}" --out "{WORK}/CP6_AGENT_RESULTS_preview.md"
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK}/CP6_AGENT_RESULTS_preview.md").read()[:12000]))

In [ ]:
# Record this session's timings on Drive (kept across sessions), then zip ALL results so far (agent files of every finished part,
# evidence summaries, timings, model digests) and the gzipped traces, and download it. No prompts, no code, no models, no embeddings go
# into the zip. Refuses to build a zip if THIS part produced no result file. The zip from the last part holds everything; hand that one
# (cp6_results.zip) to the next Claude Code session.
import zipfile, glob
old = json.load(open(TIMINGS)) if os.path.exists(TIMINGS) else {"sessions": []}
old["sessions"].append({"part": PART, "arm": ARM, "model": RUN_MODEL, "gpu": GPU, "evidence_restored": RESTORED,
                        "models": MODEL_INFO, "timings": T.rows})
json.dump(old, open(TIMINGS, "w"), indent=1)
alias = models_config.resolve(RUN_MODEL)["alias"]
mine = f"{OUT}/agent__{ARM}__{alias}.json"
if not os.path.exists(mine):
    raise SystemExit(f"STOP - PART {PART} is not complete ({mine} is missing): run the same PART again in a new session.")
json.dump({"sessions": old["sessions"], "dry_run": DRY_RUN, "context_tokens": CONTEXT_TOKENS, "limit": LIMIT,
           "graph_tables_from_zip": os.path.exists(GRAPH_ZIP),
           "evidence_vs_committed": {k: {kk: len(vv) for kk, vv in v.items()} for k, v in cmp.items()}},
          open(f"{OUT}/colab_run.json", "w"), indent=1)
done = {p: os.path.exists(f"{OUT}/agent__{a}__{models_config.resolve('fake' if DRY_RUN else m)['alias']}.json") for p, (a, m) in PARTS.items()}
zpath = f"{WORK}/cp6_results{SUFFIX}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(glob.glob(f"{OUT}/*.json")):
        z.write(p, "cp6/" + os.path.basename(p))
    for n in ("D4_materialize.json", "agent_evidence_colab.json"):
        if os.path.exists(f"{WORK}/{n}"): z.write(f"{WORK}/{n}", f"cp6/{n.replace('D4_materialize.json', 'D4_materialize_colab.json')}")
    for p in sorted(glob.glob(f"{TRACES}/*/*.json.gz")):
        z.write(p, "traces/" + os.path.relpath(p, TRACES))
print(f"PART {PART} complete. Parts done: {[p for p, d in done.items() if d]}; still to run: {[p for p, d in done.items() if not d]}")
print(zpath, os.path.getsize(zpath) // 1024, "KiB")
from google.colab import files
files.download(zpath)